In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
import json
from datetime import datetime
from collections import defaultdict

SIGNALS = {
    "Не хватает времени": ["успевать", "готовить времени нет", "быстрый обед", "перекус на ходу", "нет времени", "быстро поесть", "на бегу"],
    "Следит за фигурой": ["калории", "белок", "бжу", "жирно", "диета", "пп", "правильное питание", "зож", "похудеть", "фигура"],
    "Работает в офисе": ["офис", "кулер", "доставка в офис", "коллега", "обеденный перерыв", "работа", "бизнес-ланч", "офисный"],
    "Белок/спорт": ["протеин", "курица", "гречка", "рис", "творог", "тренировка", "фитнес", "зал", "мышцы"],
    "Экономия": ["дорого", "дешево", "цена", "бюджет", "экономить", "скидка", "акция"],
    "Фастфуд": ["бургер", "пицца", "картошка фри", "фастфуд", "макдональдс", "кфс"]
}

def parse_pikabu_posts(url, max_posts=30):
    """Парсит посты с Пикабу и собирает информацию"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
    }

    collected_posts = []

    try:
        print(f"Загружаем страницу: {url}")
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        posts_found = []

        posts_found = soup.find_all('article', class_=re.compile('story|post'))

        if not posts_found:
            posts_found = soup.find_all('div', class_=re.compile('story__|post__'))

        if not posts_found:
            posts_found = soup.find_all(['article', 'div'], class_=True)
            posts_found = [p for p in posts_found if p.find(['h1', 'h2', 'h3'])]

        print(f"Найдено {len(posts_found)} потенциальных постов")

        for idx, post in enumerate(posts_found[:max_posts]):
            try:
                title_tag = post.find(['h1', 'h2', 'h3', 'a', 'span'],
                                     class_=re.compile('title|header|heading'))
                if not title_tag:
                    title_tag = post.find(['a', 'span'], string=re.compile(r'.{10,}'))

                title = title_tag.text.strip() if title_tag else ""
                if len(title) < 10:
                    continue

                content_tag = post.find(['div', 'p', 'span'],
                                       class_=re.compile('content|text|body|description'))
                content = content_tag.text.strip() if content_tag else title

                likes = "0"
                likes_tag = post.find(class_=re.compile('likes|rating|votes'))
                if likes_tag:
                    likes_raw = likes_tag.text.strip()
                    likes = re.search(r'\d+', likes_raw).group() if re.search(r'\d+', likes_raw) else "0"

                date_tag = post.find(['time', 'span'], class_=re.compile('date|time'))
                date = date_tag.text.strip() if date_tag else datetime.now().strftime("%Y-%m-%d")

                link_tag = post.find('a', href=re.compile(r'/story/|/post/'))
                link = link_tag['href'] if link_tag else ""
                if link and not link.startswith('http'):
                    link = "https://pikabu.ru" + link

                full_text = f"{title} {content}".lower()
                matched_signals = []

                for category, keywords in SIGNALS.items():
                    if any(keyword in full_text for keyword in keywords):
                        matched_signals.append(category)

                post_data = {
                    'title': title,
                    'content_preview': content[:200] + "...",
                    'likes': likes,
                    'date': date,
                    'url': link,
                    'signals': ", ".join(matched_signals) if matched_signals else "Нет сигналов",
                    'signal_count': len(matched_signals)
                }
                collected_posts.append(post_data)

                if matched_signals:
                    print(f"\nПОСТ {idx+1}:")
                    print(f"   Заголовок: {title[:60]}...")
                    print(f"   Сигналы: {', '.join(matched_signals)}")
                    print(f"   Лайков: {likes}")

            except Exception as e:
                print(f"Ошибка при обработке поста {idx}: {e}")
                continue

            time.sleep(0.5)

    except requests.RequestException as e:
        print(f"Ошибка загрузки страницы: {e}")
    except Exception as e:
        print(f"Непредвиденная ошибка: {e}")

    return collected_posts

def analyze_collected_data(posts):
    """Анализирует собранные данные и выдает статистику"""
    if not posts:
        print("Нет данных для анализа")
        return

    print("\n")
    print("АНАЛИЗ ЦЕЛЕВОЙ АУДИТОРИИ ПО ФОРУМАМ")

    signal_counter = defaultdict(int)
    for post in posts:
        if post['signals'] != "Нет сигналов":
            signals = post['signals'].split(", ")
            for signal in signals:
                signal_counter[signal] += 1

    print("\nТОП-5 БОЛЕЙ И ПОТРЕБНОСТЕЙ АУДИТОРИИ:")
    sorted_signals = sorted(signal_counter.items(), key=lambda x: x[1], reverse=True)
    for signal, count in sorted_signals[:5]:
        percentage = (count / len(posts)) * 100
        print(f"{signal}: {count} упоминаний ({percentage:.1f}%)")

    print("\nСАМЫЕ РЕЛЕВАНТНЫЕ ПОСТЫ (несколько сигналов сразу):")
    top_posts = sorted(posts, key=lambda x: x['signal_count'], reverse=True)[:3]
    for post in top_posts:
        if post['signal_count'] > 0:
            print(f"\n{post['title'][:70]}...")
            print(f" Сигналы: {post['signals']}")
            print(f" Лайков: {post['likes']}")
            if post['url']:
                print(f" Ссылка: {post['url']}")

    total_likes = sum(int(post['likes']) for post in posts if post['likes'].isdigit())
    avg_likes = total_likes // len(posts) if posts else 0

    print("\nОБЩАЯ СТАТИСТИКА:")
    print(f"   Всего проанализировано постов: {len(posts)}")
    print(f"   Среднее количество лайков: {avg_likes}")
    print(f"   Посты с сигналами: {sum(1 for p in posts if p['signal_count'] > 0)} из {len(posts)}")

    filename = f"pikabu_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(posts, f, ensure_ascii=False, indent=2)
    print(f"\nДанные сохранены в файл: {filename}")
    try:
        import csv
        csv_filename = filename.replace('.json', '.csv')
        with open(csv_filename, 'w', encoding='utf-8-sig', newline='') as f:
            fieldnames = ['title', 'likes', 'date', 'signals', 'signal_count', 'url']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for post in posts:
                row = {key: post.get(key, '') for key in fieldnames}
                writer.writerow(row)
    except Exception as e:
        print(f"Ошибка при сохранении CSV: {e}")

def parse_multiple_pages():
    """Парсит несколько страниц с разными тегами"""
    topics = [
        ("Обед", "https://pikabu.ru/tag/Обед/hot"),
        ("ПП", "https://pikabu.ru/tag/ПП/hot"),
        ("Фитнес", "https://pikabu.ru/tag/Фитнес/hot"),
        ("Доставка еды", "https://pikabu.ru/tag/Доставка%20еды/hot"),
        ("ЗОЖ", "https://pikabu.ru/tag/ЗОЖ/hot"),
        ("Еда", "https://pikabu.ru/tag/Еда/hot"),
    ]
    all_posts = []

    for topic_name, url in topics:
        print(f"\n")
        print(f"Парсинг темы: {topic_name}")
        posts = parse_pikabu_posts(url, max_posts=15)
        all_posts.extend(posts)
        print(f"Собрано {len(posts)} постов по теме '{topic_name}'")
        time.sleep(2)

    print(f"\n")
    print(f"ИТОГО СОБРАНО: {len(all_posts)} ПОСТОВ")
    analyze_collected_data(all_posts)

    return all_posts

if __name__ == "__main__":
    print("ЗАПУСК ПАРСИНГА ФОРУМОВ ДЛЯ АНАЛИЗА ЦА САЛАТ-БАРА")

    all_posts = parse_multiple_pages()

ЗАПУСК ПАРСИНГА ФОРУМОВ ДЛЯ АНАЛИЗА ЦА САЛАТ-БАРА


Парсинг темы: Обед
Загружаем страницу: https://pikabu.ru/tag/Обед/hot
Найдено 10 потенциальных постов

ПОСТ 1:
   Заголовок: ГОТОВИМ ВКУСНЕЙШИЙ УЖИН ДЛЯ ВСЕЙ СЕМЬИ⁠⁠...
   Сигналы: Следит за фигурой, Экономия
   Лайков: 0

ПОСТ 3:
   Заголовок: Щи «Пощады не будет»⁠⁠...
   Сигналы: Следит за фигурой, Белок/спорт, Экономия
   Лайков: 17

ПОСТ 5:
   Заголовок: Тушёная пекинская капуста⁠⁠...
   Сигналы: Следит за фигурой, Белок/спорт
   Лайков: 10

ПОСТ 6:
   Заголовок: Мужская кухня. Суп из индейки, с вешенками и фунчозой⁠⁠...
   Сигналы: Следит за фигурой, Белок/спорт
   Лайков: 17

ПОСТ 8:
   Заголовок: Обзор продуктов с повышенным содержанием протеина⁠⁠...
   Сигналы: Белок/спорт
   Лайков: 565

ПОСТ 10:
   Заголовок: Ответ на пост «"Суп - выручайка: когда нет времени на бульон...
   Сигналы: Не хватает времени, Следит за фигурой, Белок/спорт
   Лайков: 30
Собрано 8 постов по теме 'Обед'


Парсинг темы: ПП
Загружаем страницу: https:/

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
import json
from datetime import datetime
from collections import defaultdict

SIGNALS = {
    "Не хватает времени": ["нет времени", "быстрый", "готовить некогда", "экономия времени", "не надо готовить", "освободилось время"],
    "Следит за фигурой/ПП": ["пп", "зож", "калории", "бжу", "диета", "фигура", "похудеть", "дефицит калорий", "правильное питание"],
    "Работает в офисе": ["офис", "работа", "коллеги", "бизнес-ланч", "обед на работе", "офисный работник"],
    "Фитнес/спорт": ["фитнес", "тренировка", "зал", "белок", "протеин", "мышцы", "спорт"],
    "Проблемы с доставкой": ["доставка", "курьер", "опоздали", "время доставки", "задержалась"],
    "Качество еды": ["вкусно", "невкусно", "свежий", "прокис", "качество", "порции"],
    "Цена/бюджет": ["дорого", "дешево", "бюджет", "цена", "экономия", "тысяч"]
}

def parse_irecommend_category(category_url, max_reviews=20):
    """
    Парсит отзывы из категории IRecommend
    Использует прямые URL категорий вместо поиска
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    collected_reviews = []

    try:
        print(f"Загружаем: {category_url}")
        response = requests.get(category_url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, 'html.parser')

        review_links = []

        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/content/' in href and href not in review_links:
                if href.startswith('/content/'):
                    full_link = "https://irecommend.ru" + href
                    review_links.append(full_link)

        review_blocks = soup.find_all('div', class_=re.compile('review|content-item|product-review'))
        for block in review_blocks:
            link = block.find('a', href=re.compile(r'/content/'))
            if link and link['href']:
                full_link = "https://irecommend.ru" + link['href']
                if full_link not in review_links:
                    review_links.append(full_link)

        print(f"Найдено ссылок: {len(review_links)}")

        review_links = review_links[:max_reviews]

        for idx, link in enumerate(review_links):
            try:
                time.sleep(1.5)

                review_response = requests.get(link, headers=headers, timeout=15)
                review_soup = BeautifulSoup(review_response.text, 'html.parser')

                title_tag = review_soup.find('h1')
                title = title_tag.text.strip() if title_tag else "Без заголовка"

                content = ""
                content_selectors = [
                    ('div', {'class': re.compile(r'field-type-text-with-summary|content|body')}),
                    ('div', {'itemprop': 'reviewBody'}),
                    ('div', {'class': 'content clearfix'})
                ]

                for tag, attrs in content_selectors:
                    content_tag = review_soup.find(tag, attrs)
                    if content_tag:
                        content = content_tag.text.strip()
                        break

                if not content:
                    for div in review_soup.find_all('div'):
                        if len(div.text) > 200 and 'отзыв' not in div.text.lower():
                            content = div.text.strip()
                            break

                author_tag = review_soup.find('span', class_=re.compile('username|author'))
                if not author_tag:
                    author_tag = review_soup.find('a', class_=re.compile('user'))
                author = author_tag.text.strip() if author_tag else "Аноним"

                date_tag = review_soup.find('span', class_=re.compile('submitted|date|time'))
                if not date_tag:
                    date_tag = review_soup.find('time')
                date = date_tag.text.strip() if date_tag else ""

                rating = 0
                rating_tag = review_soup.find('span', class_=re.compile('rating|stars'))
                if rating_tag:
                    stars = rating_tag.find_all('span', class_=re.compile('star'))
                    rating = sum(1 for star in stars if 'full' in str(star) or 'active' in str(star))

                full_text = f"{title} {content[:1000]}".lower()
                matched_signals = []

                for category, keywords in SIGNALS.items():
                    for keyword in keywords:
                        if keyword.lower() in full_text:
                            matched_signals.append(category)
                            break
                matched_signals = list(set(matched_signals))
                review_data = {
                    'source': 'IRecommend',
                    'url': link,
                    'title': title[:100],
                    'content': content[:700] + "..." if len(content) > 700 else content,
                    'author': author,
                    'date': date,
                    'rating': rating,
                    'signals': ", ".join(matched_signals) if matched_signals else "Нет сигналов",
                    'signal_count': len(matched_signals)
                }
                collected_reviews.append(review_data)

                if matched_signals:
                    print(f"\n[{idx+1}] {title[:50]}...")
                    print(f"Сигналы: {', '.join(matched_signals)}")
                    print(f"Рейтинг: {rating}/5 звезд")
                else:
                    print(f"[{idx+1}] {title[:40]}... (сигналов нет)")

            except Exception as e:
                print(f"Ошибка: {e}")
                continue

        return collected_reviews
    except Exception as e:
        print(f"Ошибка: {e}")
        return []

def parse_specific_reviews():
    specific_urls = [
        "https://irecommend.ru/content/pravilnoe-pitanie-s-dostavkoi-na-dom-ot-daily-food-umerennyi-defitsit-kalorii-i-sbros-lishne",
        "https://irecommend.ru/content/ochen-udobno-404",
        "https://irecommend.ru/content/blestyashche-72",
        "https://irecommend.ru/content/vkusno-1881",
        "https://irecommend.ru/content/poprobovat-edu-tak-i-ne-udalos",
        "https://irecommend.ru/content/ne-rekomenduyu-2572",
        "https://irecommend.ru/content/uzhasnoe-kachestvo-i-obman-0",
    ]
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    collected_reviews = []

    print("Парсинг конкретных отзывов о доставках еды")
    for idx, url in enumerate(specific_urls):
        try:
            time.sleep(1.5)
            response = requests.get(url, headers=headers, timeout=15)
            soup = BeautifulSoup(response.text, 'html.parser')
            title_tag = soup.find('h1')
            title = title_tag.text.strip() if title_tag else "Без заголовка"

            content = ""
            content_selectors = [
                'div[itemprop="reviewBody"]',
                'div.field-type-text-with-summary',
                'div.content',
                'div.clearfix'
            ]

            for selector in content_selectors:
                content_tag = soup.select_one(selector)
                if content_tag:
                    content = content_tag.text.strip()
                    break
            if not content:
                for div in soup.find_all('div'):
                    if len(div.text) > 300:
                        content = div.text.strip()
                        break

            author_tag = soup.find('span', class_=re.compile('username|author'))
            author = author_tag.text.strip() if author_tag else "Аноним"
            date_tag = soup.find('span', class_=re.compile('submitted|date'))
            date = date_tag.text.strip() if date_tag else ""

            rating = 0
            rating_stars = soup.find_all('span', class_='star')
            rating = sum(1 for star in rating_stars if 'full' in str(star) or 'active' in str(star))

            full_text = f"{title} {content}".lower()
            matched_signals = []

            for category, keywords in SIGNALS.items():
                for keyword in keywords:
                    if keyword.lower() in full_text:
                        matched_signals.append(category)
                        break

            matched_signals = list(set(matched_signals))

            review_data = {
                'source': 'IRecommend (direct)',
                'url': url,
                'title': title[:100],
                'content': content[:800] + "..." if len(content) > 800 else content,
                'author': author,
                'date': date,
                'rating': rating,
                'signals': ", ".join(matched_signals) if matched_signals else "Нет сигналов",
                'signal_count': len(matched_signals)
            }

            collected_reviews.append(review_data)
        except Exception as e:
            print(f"Ошибка: {e}")
            continue
    return collected_reviews

def analyze_reviews(reviews):
    if not reviews:
        print("Нет данных для анализа")
        return

    signal_counter = defaultdict(int)
    for review in reviews:
        if review['signals'] != "Нет сигналов":
            for signal in review['signals'].split(", "):
                signal_counter[signal] += 1

    print("РЕЗУЛЬТАТЫ АНАЛИЗА ОТЗЫВОВ")

    print("\nТОП ПОТРЕБНОСТЕЙ АУДИТОРИИ:")
    for signal, count in sorted(signal_counter.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / len(reviews)) * 100
        print(f"   - {signal}: {count} упоминаний ({percentage:.1f}%)")

    print("\nКлючевые выводы")

    if signal_counter.get("Не хватает времени", 0) > 0:
        print("Люди ценят экономию времени")
    if signal_counter.get("Следит за фигурой/ПП", 0) > 0:
        print("Аудитория следит за питанием")
    if signal_counter.get("Фитнес/спорт", 0) > 0:
        print("Есть запрос на спортивное питание")
    if signal_counter.get("Работает в офисе", 0) > 0:
        print("Спрос со стороны офисных сотрудники")
    if signal_counter.get("Цена/бюджет", 0) > 0:
        print("Цена важна для потребителей, нужны бюджетные тарифы или акции")

    filename = f"irecommend_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(reviews, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено в: {filename}")

print("ПАРСИНГ ОТЗЫВОВ IRecommend ДЛЯ АНАЛИЗА ЦА")

all_reviews = []
direct_reviews = parse_specific_reviews()
all_reviews.extend(direct_reviews)

print(f"\nВсего собрано отзывов: {len(all_reviews)}")
analyze_reviews(all_reviews)

ПАРСИНГ ОТЗЫВОВ IRecommend ДЛЯ АНАЛИЗА ЦА
Парсинг конкретных отзывов о доставках еды

Всего собрано отзывов: 7
РЕЗУЛЬТАТЫ АНАЛИЗА ОТЗЫВОВ

ТОП ПОТРЕБНОСТЕЙ АУДИТОРИИ:
   - Проблемы с доставкой: 6 упоминаний (85.7%)
   - Качество еды: 4 упоминаний (57.1%)
   - Фитнес/спорт: 2 упоминаний (28.6%)
   - Не хватает времени: 2 упоминаний (28.6%)
   - Цена/бюджет: 2 упоминаний (28.6%)
   - Работает в офисе: 1 упоминаний (14.3%)
   - Следит за фигурой/ПП: 1 упоминаний (14.3%)

Ключевые выводы
Люди ценят экономию времени
Аудитория следит за питанием
Есть запрос на спортивное питание
Спрос со стороны офисных сотрудники
Цена важна для потребителей, нужны бюджетные тарифы или акции

Сохранено в: irecommend_analysis_20260517_0822.json


In [4]:
!apt-get update
!apt-get install -y wget unzip xvfb libxi6 libgconf-2-4
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
!apt-get update
!apt-get install -y google-chrome-stable

Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Get:2 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:10 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,208 B]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,950 B in 1s (5,103 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/dists/stable/InRelease: Key is stored in legacy trusted.gpg key

In [3]:
!apt-get install -y -qq chromium-browser chromium-chromedriver

Preconfiguring packages ...
Selecting previously unselected package apparmor.
(Reading database ... 118842 files and directories currently installed.)
Preparing to unpack .../apparmor_3.0.4-2ubuntu2.5_amd64.deb ...
Unpacking apparmor (3.0.4-2ubuntu2.5) ...
Selecting previously unselected package squashfs-tools.
Preparing to unpack .../squashfs-tools_1%3a4.5-3build1_amd64.deb ...
Unpacking squashfs-tools (1:4.5-3build1) ...
Selecting previously unselected package udev.
Preparing to unpack .../udev_249.11-0ubuntu3.20_amd64.deb ...
Unpacking udev (249.11-0ubuntu3.20) ...
Selecting previously unselected package libfuse3-3:amd64.
Preparing to unpack .../libfuse3-3_3.10.5-1build1_amd64.deb ...
Unpacking libfuse3-3:amd64 (3.10.5-1build1) ...
Selecting previously unselected package snapd.
Preparing to unpack .../snapd_2.74.1+ubuntu22.04.4_amd64.deb ...
Unpacking snapd (2.74.1+ubuntu22.04.4) ...
Setting up apparmor (3.0.4-2ubuntu2.5) ...
Created symlink /etc/systemd/system/sysinit.target.wants/

In [6]:
!pip install selenium -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.0 MB/s eta 0:00:00


In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import json
from datetime import datetime
from collections import defaultdict

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

SIGNALS = {
    "Не хватает времени": [
        "нет времени", "быстрый", "готовить некогда", "быстро поесть",
        "перекус", "цейтнот", "быстро", "некогда", "экономия времени"
    ],
    "Следит за фигурой/ПП": [
        "пп", "зож", "калории", "бжу", "диета", "фигура", "похудеть",
        "правильное питание", "полезно", "диетическое", "лёгкий", "кбжу"
    ],
    "Работает в офисе": [
        "офис", "работа", "коллеги", "бизнес-ланч", "обед на работе",
        "бизнес-центр", "офисный", "доставка в офис"
    ],
    "Фитнес/спорт": [
        "фитнес", "тренировка", "зал", "белок", "протеин", "спорт",
        "тренируюсь", "после тренировки", "спортзал"
    ],
    "Качество еды": [
        "вкусно", "невкусно", "свежий", "прокис", "качество", "порции",
        "холодный", "вкус", "безвкусно", "свежесть"
    ],
    "Цена/бюджет": [
        "дорого", "дешево", "цена", "экономия", "скидка", "акция",
        "бюджетно", "ценник", "стоимость"
    ],
    "Проблемы с сервисом": [
        "обслуживание", "персонал", "хам", "долго", "очередь",
        "медленно", "не убрали", "грязно"
    ]
}

search_queries = [
    "фитнес клуб",
    "салат бар",
    "здоровое питание",
    "пп кафе",
    "бизнес ланч"
]

def search_and_parse_flamp(query, max_places=5, max_reviews_per_place=15):
    all_reviews = []
    search_url = f"https://flamp.ru/search/{query.replace(' ', '%20')}"
    print(f"\n🔍 Поиск: {query}")
    print(f"   URL: {search_url}")

    try:
        driver.get(search_url)
        time.sleep(5)

        for _ in range(2):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        place_links = []
        selectors = [
            "a[href*='/firm/']",
            "div[class*='item'] a",
            "a[class*='title']"
        ]
        for selector in selectors:
            links = driver.find_elements(By.CSS_SELECTOR, selector)
            for link in links:
                href = link.get_attribute('href')
                if href and '/firm/' in href and href not in place_links:
                    place_links.append(href)
            if place_links:
                print(f"Найдено ссылок: {len(place_links)}")
                break

        if not place_links:
            print("Организации не найдены")
            return all_reviews

        place_links = place_links[:max_places]
        for idx, firm_url in enumerate(place_links):
            try:
                print(f"\n[{idx+1}/{len(place_links)}] Загружаем: {firm_url}")
                driver.get(firm_url)
                time.sleep(4)

                for scroll in range(5):
                    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(2)
                    print(f"Прокрутка {scroll+1}/5", end="\r")

                print()
                review_elements = []
                review_selectors = [
                    "div[class*='review']",
                    "div[class*='Review']",
                    "div[data-testid='review-card']"
                ]

                for selector in review_selectors:
                    review_elements = driver.find_elements(By.CSS_SELECTOR, selector)
                    if review_elements:
                        print(f"Найдено отзывов: {len(review_elements)}")
                        break

                if not review_elements:
                    print(f"Отзывы не найдены")
                    continue

                org_name = ""
                name_elem = driver.find_elements(By.CSS_SELECTOR, "h1, div[class*='name']")
                if name_elem:
                    org_name = name_elem[0].text.strip()[:50]
                review_elements = review_elements[:max_reviews_per_place]

                for rev_idx, elem in enumerate(review_elements):
                    try:
                        text = elem.text.strip()

                        if len(text) < 30:
                            continue
                        author = "Аноним"
                        author_elem = elem.find_elements(By.CSS_SELECTOR, "span[class*='name'], div[class*='author']")
                        if author_elem:
                            author = author_elem[0].text.strip()[:40]
                        rating = 0
                        rating_elem = elem.find_elements(By.CSS_SELECTOR, "span[class*='star'], div[class*='rating']")
                        for r in rating_elem:
                            if 'active' in r.get_attribute('class') or 'filled' in r.get_attribute('class'):
                                rating += 1

                        text_lower = text.lower()
                        matched_signals = []

                        for category, keywords in SIGNALS.items():
                            for keyword in keywords:
                                if keyword in text_lower:
                                    matched_signals.append(category)
                                    break

                        matched_signals = list(set(matched_signals))
                        if matched_signals:
                            review_data = {
                                "query": query,
                                "organization": org_name,
                                "author": author,
                                "rating": rating,
                                "text": text[:500],
                                "signals": ", ".join(matched_signals)
                            }
                            all_reviews.append(review_data)

                            star_display = rating if rating > 0 else "?"
                            print(f"\n[{rev_idx+1}] {author} {star_display}")
                            print(f"{text[:100]}...")
                            print(f"{', '.join(matched_signals)}")

                    except Exception as e:
                        continue

                time.sleep(2)

            except Exception as e:
                print(f"Ошибка при загрузке: {e}")
                continue

    except Exception as e:
        print(f"Ошибка поиска: {e}")

    return all_reviews

print("FLAMP.RU")
print(f"\nПоисковые запросы: {', '.join(search_queries)}")
print(f"Целевая аудитория: люди, которые ходят в фитнес и следят за питанием")

all_reviews = []

for query in search_queries:
    reviews = search_and_parse_flamp(query, max_places=4, max_reviews_per_place=15)
    all_reviews.extend(reviews)
    print(f"\nИтого по запросу '{query}': {len(reviews)} отзывов с сигналами")
    time.sleep(3)

driver.quit()

print("ОБЩИЙ АНАЛИЗ СОБРАННЫХ ДАННЫХ")


if not all_reviews:
    print(" Отзывы не найдены")
    print("\nВозможные причины:")
    print("1. Сайт временно недоступен")
    print("2. Нет отзывов по этим запросам")
else:
    signal_counter = defaultdict(int)
    for review in all_reviews:
        for signal in review["signals"].split(", "):
            signal_counter[signal] += 1

    print(f"\nСтатистика:")
    print(f"Всего собрано отзывов: {len(all_reviews)}")
    print(f"Уникальных организаций: {len(set(r['organization'] for r in all_reviews))}")

    print(f"\nТОП ПОТРЕБНОСТЕЙ И ПРОБЛЕМ АУДИТОРИИ:")
    sorted_signals = sorted(signal_counter.items(), key=lambda x: x[1], reverse=True)
    for signal, count in sorted_signals:
        percentage = (count / len(all_reviews)) * 100
        bar_length = int(percentage / 2)
        bar = "█" * bar_length + "░" * (20 - bar_length)
        print(f"   {signal:22} | {bar} {count} ({percentage:.0f}%)")

    print(f"\nКЛЮЧЕВЫЕ ВЫВОДЫ:")

    insights_list = []
    if signal_counter.get("Не хватает времени", 0) > 0:
        insights_list.append("Делаем акцент на быстроте")
    if signal_counter.get("Следит за фигурой/ПП", 0) > 0:
        insights_list.append("Указываем КБЖУ на каждом блюде")
    if signal_counter.get("Фитнес/спорт", 0) > 0:
        insights_list.append("Добавляем высокобелковые опции")
    if signal_counter.get("Работает в офисе", 0) > 0:
        insights_list.append("Предлагаем бизнес-ланчи")
    if signal_counter.get("Качество еды", 0) > 0:
        insights_list.append("Делаем упор на свежесть ингридиентов")
    if signal_counter.get("Цена/бюджет", 0) > 0:
        insights_list.append("Добавляем бюджетные позиции и программу лояльности")
    if signal_counter.get("Проблемы с сервисом", 0) > 0:
        insights_list.append("Обучаем персонал вежливости, добавляем кассы самообмлуживания")

    for insight in insights_list:
        print(f"{insight}")

    if not insights_list:
        print("Соберите больше отзывов для точных выводов")

    print(f"\nПРИМЕРЫ ОТЗЫВОВ:")
    signal_reviews = [r for r in all_reviews if r['signals'] != "Нет сигналов"]
    for i, review in enumerate(signal_reviews[:5], 1):
        print(f"\n   {i}. {review['organization']}")
        print(f"   {review['author']} {review['rating']}")
        print(f"   {review['text'][:150]}...")
        print(f"   Сигналы: {review['signals']}")

    filename = f"flamp_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_reviews, f, ensure_ascii=False, indent=2)
    print(f"\nСохранено в: {filename}")

    txt_filename = filename.replace('.json', '_report.txt')
    with open(txt_filename, 'w', encoding='utf-8') as f:
        f.write("ОТЧЕТ ПО АНАЛИЗУ ОТЗЫВОВ FLAMP.RU\n")
        f.write(f"Всего отзывов: {len(all_reviews)}\n")
        f.write(f"Отзывов с сигналами: {len(signal_reviews)}\n\n")
        f.write("ТОП ПОТРЕБНОСТЕЙ:\n")
        for signal, count in sorted_signals:
            f.write(f"  {signal}: {count}\n")
        f.write("\n")
        f.write("ПРИМЕРЫ ОТЗЫВОВ:\n\n")
        for review in signal_reviews[:10]:
            f.write(f"Организация: {review['organization']}\n")
            f.write(f"Автор: {review['author']}\n")
            f.write(f"Рейтинг: {review['rating']}\n")
            f.write(f"Сигналы: {review['signals']}\n")
            f.write(f"Текст: {review['text']}\n")
            f.write("\n\n")

    print(f"Текстовый отчет: {txt_filename}")

print(f"\nВсего обработано отзывов: {len(all_reviews)}")

FLAMP.RU

Поисковые запросы: фитнес клуб, салат бар, здоровое питание, пп кафе, бизнес ланч
Целевая аудитория: люди, которые ходят в фитнес и следят за питанием

🔍 Поиск: фитнес клуб
   URL: https://flamp.ru/search/фитнес%20клуб
Найдено ссылок: 10

[1/4] Загружаем: https://moscow.flamp.ru/firm/oblaka_fitnes_klub-70000001048491159
Прокрутка 5/5
Найдено отзывов: 12

[3] Аноним ?
Красивый и хорошо оснащённый зал. Все тренажёры современные и насколько я понимаю в этом, самых попу...
Фитнес/спорт

[2/4] Загружаем: https://moscow.flamp.ru/firm/oblaka_fitnes_klub-70000001044536478
Прокрутка 5/5
Найдено отзывов: 13

[6] Аноним ?
Первое занятие прошло отлично, я выбрала годовой абонемент с полным доступом. Для меня было важно, ч...
Качество еды, Работает в офисе, Фитнес/спорт

[3/4] Загружаем: https://moscow.flamp.ru/firm/oblaka_fitnes_klub-70000001030248940
Прокрутка 5/5
Найдено отзывов: 13

[12] Аноним ?
Я занимаюсь в этом фитнес-клубе по абонементу, и мне особенно нравятся тренеры Юлия, мужч